# 1. Імпорт бібліотек

In [ ]:
import torch
import pandas as pd
import numpy as np
import evaluate
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

print(f"PyTorch версія: {torch.__version__}")
print(f"CUDA доступна: {torch.cuda.is_available()}")

# 2. Конфігурація та завантаження даних

In [ ]:
FILE_PATH = "../../data/final_dataset.csv"
MODEL_NAME = "youscan/ukr-roberta-base"
TEXT_COLUMN = "text"
LABEL_COLUMN = "fake"
MAX_LENGTH = 256

print(f"Завантаження даних з {FILE_PATH}...")
df = pd.read_csv(FILE_PATH, encoding='utf-8')

df = df.rename(columns={LABEL_COLUMN: 'label'})
df = df[[TEXT_COLUMN, 'label']]

dataset = Dataset.from_pandas(df)

print("Дані завантажено:")
print(dataset)

# 3. Розділення даних (70/15/15)

In [ ]:
train_test_split = dataset.train_test_split(test_size=0.3, seed=42)
test_valid_split = train_test_split['test'].train_test_split(test_size=0.5, seed=42)

dataset_dict = DatasetDict({
    'train': train_test_split['train'],
    'validation': test_valid_split['train'],
    'test': test_valid_split['test']
})

print("Дані розділено:")
print(dataset_dict)

# 4. Завантаження моделі та токенізатора

In [ ]:
print(f"Завантаження токенізатора для {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(f"Завантаження моделі {MODEL_NAME}...")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, 
    num_labels=2
)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
print(f"Модель завантажена на: {device}")

# 5. Токенізація

In [ ]:
def tokenize_function(examples):
    return tokenizer(
        examples[TEXT_COLUMN], 
        padding="max_length", 
        truncation=True, 
        max_length=MAX_LENGTH
    )

print("Токенізація датасету...")
tokenized_datasets = dataset_dict.map(tokenize_function, batched=True)
tokenized_datasets = tokenized_datasets.remove_columns([TEXT_COLUMN])
print("Токенізація завершена.")

# 6. Функція для розрахунку метрик

In [ ]:
from sklearn.metrics import fbeta_score

accuracy_metric = evaluate.load("accuracy")
precision_metric = evaluate.load("precision")
recall_metric = evaluate.load("recall")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    
    accuracy = accuracy_metric.compute(predictions=predictions, references=labels)
    precision = precision_metric.compute(predictions=predictions, references=labels)
    recall = recall_metric.compute(predictions=predictions, references=labels)
    f2 = fbeta_score(labels, predictions, beta=2)
    
    return {
        "accuracy": accuracy["accuracy"],
        "precision": precision["precision"],
        "recall": recall["recall"],
        "f2": f2
    }

print("Функція метрик готова.")

# 7. Налаштування та запуск навчання

In [ ]:
training_args = TrainingArguments(
    output_dir="../../data/results_ukrbert",
    num_train_epochs=5, 
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    per_device_eval_batch_size=4,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_steps=100,
    push_to_hub=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

print("Починаємо тренування...")
trainer.train()
print("Тренування завершено.")

# 8. Оцінка на тестовій вибірці

In [ ]:
print(" ОЦІНКА НА ТЕСТОВІЙ ВИБІРЦІ ".center(50, "="))

test_results = trainer.evaluate(eval_dataset=tokenized_datasets["test"])

print("Результати на тестовій вибірці:\n")
print(f"Accuracy:  {test_results['eval_accuracy']:.4f}")
print(f"Precision: {test_results['eval_precision']:.4f}")
print(f"Recall:    {test_results['eval_recall']:.4f}")
print(f"F2-score:  {test_results['eval_f2']:.4f}")